# Impulse and Torque MLP (simple)

This notebook trains a physics-structured and physics informed MLP for a rigid body interacting with a plane-
As input we get the rotation as a 6-D rotation Matrix (with cos and sin)
We parameterise normal force with Hooke, similar to our simulations
We internally predict the contact normal
We couple torque to force via cross product: torque = r_{lever} x f
In the loss we use a Hube loss with a weight for energy conservation

In [39]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import json
import math
from tqdm import tqdm

In [40]:
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")

device = get_device()
print("Using device:", device)

Using device: mps


## Constants

In [41]:
TRAIN_TEST_SPLIT = 0.8

## Colab Only - Download data

In [42]:
def is_colab():
    try:
        import google.colab
        return True
    except Exception as e:
        return False

if is_colab():

    from google.colab import drive
    from tqdm import tqdm
    import os

    drive.mount('/content/drive')

    src_path = "/content/drive/MyDrive/final_output_contact_points.json"
    dst_path = "/content/final_output_contact_points.json"


    chunk_size = 1024 * 1024  # 1 MB
    file_size = os.path.getsize(src_path)

    with open(src_path, 'rb') as src, open(dst_path, 'wb') as dst:
        with tqdm(total=file_size, unit='B', unit_scale=True, desc="Copying to /content") as pbar:
            while True:
                chunk = src.read(chunk_size)
                if not chunk:
                    break
                dst.write(chunk)
                pbar.update(len(chunk))

    print("Done! File is now in:", dst_path)
    # Print the first entry
    with open(dst_path, 'r') as f:
        data = json.load(f)

    first = data[0] if isinstance(data, list) else next(iter(data.values()))
    print("\nFirst entry:")
    print(json.dumps(first, indent=2))

## Dataset

Here we get the contat points with individual forces. From these forces, we calculate the torque and sum up the forces and torques to one force and one torque

In [43]:
class ContactDataset(Dataset):
    """World-frame wrench dataset for a rigid body on a plane.

    Features (10-D):
        [v_x, v_y, v_z,                           # linear velocity (world frame)
         rel_pos_z,                               # height above plane
         sin(roll), cos(roll),
         sin(pitch), cos(pitch),
         sin(yaw), cos(yaw)]

    Targets (world frame, PHYSICAL UNITS — not normalised):
        force:  (3,)   sum of per-contact forces
        torque: (3,)   sum of (r_world - com_world) x f_world
    """

    def __init__(self, data_list):
        features, forces, torques, collisions = [], [], [], []
        lin_vels, ang_vels = [], []

        for contact in data_list:
            rel_pos = contact["relative_position_to_collider"]
            rel_rot = contact["relative_rotation_to_collider"]
            lin_vel = contact["linear_velocity"]
            ang_vel = contact["angular_velocity"]
            cube_pos = contact["self_position"]

            roll, pitch, yaw = rel_rot["roll"], rel_rot["pitch"], rel_rot["yaw"]
            feats = np.array([
                lin_vel["x"], lin_vel["y"], lin_vel["z"],
                ang_vel["x"], ang_vel["y"], ang_vel["z"],
                rel_pos["z"],
                np.sin(roll), np.cos(roll),
                np.sin(pitch), np.cos(pitch),
                np.sin(yaw), np.cos(yaw),
            ], dtype=np.float32)


            cube_pos_numpy = np.array([cube_pos["x"], cube_pos["y"], cube_pos["z"]])
            force_numpy = np.zeros(3, dtype=np.float32)
            torque_numpy = np.zeros(3, dtype=np.float32)

            for p in contact.get("points", []):
                p_force = p["force"]
                lever_pos = p["contact_position_world"]
                point_force_numpy = np.array([p_force["x"], p_force["y"], p_force["z"]], dtype=np.float32)
                lever_rel_pos = np.array([lever_pos["x"], lever_pos["y"], lever_pos["z"]], dtype=np.float32) - cube_pos_numpy

                force_numpy += point_force_numpy
                torque_numpy += np.cross(lever_rel_pos, point_force_numpy)

            features.append(feats)
            forces.append(force_numpy)
            torques.append(torque_numpy)
            collisions.append([min(len(contact.get("points", [])), 1)])
            lin_vels.append([lin_vel["x"], lin_vel["y"], lin_vel["z"]])
            ang_vels.append([ang_vel["x"], ang_vel["y"], ang_vel["z"]])

        
        self.features   = torch.FloatTensor(np.asarray(features))
        self.forces     = torch.FloatTensor(np.asarray(forces))
        self.torques    = torch.FloatTensor(np.asarray(torques))
        self.collisions = torch.FloatTensor(np.asarray(collisions))
        
        self.lin_vels   = torch.FloatTensor(np.asarray(lin_vels, dtype=np.float32))
        self.ang_vels   = torch.FloatTensor(np.asarray(ang_vels, dtype=np.float32))


    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return (
            self.features[idx],
            {
                "force":        self.forces[idx],
                "torque":       self.torques[idx],
                "is_collision": self.collisions[idx],
                "lin_vel":      self.lin_vels[idx],
                "ang_vel":      self.ang_vels[idx],
            },
        )

### Show dataset

show the first entry of the dataset

In [44]:
json_file = 'final_output_contact_points.json'
with open(json_file, 'r') as f:
    data = json.load(f)
if isinstance(data, dict):
    data = [data]

full_dataset = ContactDataset(data)


Print the first entry of the dataset

In [45]:
print(full_dataset[0])

(tensor([ 2.2536,  9.3979, -0.2112,  4.5779,  4.4005,  1.8026,  0.6932, -0.9757,
         0.2193,  0.1408,  0.9900,  0.0744,  0.9972]), {'force': tensor([9.2815e-12, 4.3170e-12, 1.7108e+01]), 'torque': tensor([-2.0003e+00,  7.9867e-01,  8.8371e-13]), 'is_collision': tensor([1.]), 'lin_vel': tensor([ 2.2536,  9.3979, -0.2112]), 'ang_vel': tensor([4.5779, 4.4005, 1.8026])})


Showing some info of the dataset

In [46]:
print("collisions:", int(full_dataset.collisions.sum().item()),
      "/", len(full_dataset))
mask = full_dataset.collisions.squeeze(-1).bool()
if mask.any():
    f_rms = full_dataset.forces[mask].pow(2).mean().sqrt().item()
    t_rms = full_dataset.torques[mask].pow(2).mean().sqrt().item()
    print(f"\nContact-only RMS force  = {f_rms:.4f}")
    print(f"Contact-only RMS torque = {t_rms:.4f}")
    print(f"Suggested w_torque / w_force ratio ≈ {f_rms / max(t_rms, 1e-8):.3f}")


collisions: 1117730 / 2147197

Contact-only RMS force  = 330.2821
Contact-only RMS torque = 82.2579
Suggested w_torque / w_force ratio ≈ 4.015


Split the dataset into train and test dataset and create data loaders

In [47]:
train_size = int(TRAIN_TEST_SPLIT * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator = torch.Generator())

use_gpu = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=1024,
    shuffle=True,
    num_workers=4 if use_gpu else 0,
    pin_memory=use_gpu,
    persistent_workers=use_gpu,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=1024,
    shuffle=False,
    num_workers=4 if use_gpu else 0,
    pin_memory=use_gpu,
    persistent_workers=use_gpu,
)

Validation checks on the dataset:

In [48]:
input_dim = full_dataset.features.shape[1]
print(f"input_dim={input_dim}  (expect 10)")
print(f"force target shape = {tuple(full_dataset.forces.shape)}  (expect (N, 3))")
print(f"torque target shape = {tuple(full_dataset.torques.shape)}  (expect (N, 3))")
assert max(full_dataset.collisions) == 1


input_dim=13  (expect 10)
force target shape = (2147197, 3)  (expect (N, 3))
torque target shape = (2147197, 3)  (expect (N, 3))


## Model

Here is our physically structured model

In [49]:
import torch
import torch.nn as nn
import torch.nn.functional as F

HEAD_OUT_DIM = 11  # 1 (collision) + 1 (depth) + 3 (force_residual) + 3 (normal) + 3 (lever)
VEL_SLICE = slice(0, 3)  # [vx, vy, vz] are the first 3 features


class ResBlock(nn.Module):
    def __init__(self, width, expansion=4):
        super().__init__()
        self.act = nn.ReLU(inplace=True)

        self.block = nn.Sequential(
            nn.LayerNorm(width),
            nn.Linear(width, width * expansion),
            self.act,
            nn.LayerNorm(width * expansion),
            nn.Linear(width * expansion, width),
            self.act,
        )

    def forward(self, x):
        return self.act(x + self.block(x))


class WrenchPredictor(nn.Module):
    """
    Predicts net wrench (force + torque) with a head whose force assembly
    matches the physics sim, plus a learned residual to absorb deviations
    the analytical model cannot express.

    Sim-matching part:
        F_spring  = -k * depth                        (depth <= 0)
        F_damping = -c * (v . n),  c = 2*sqrt(k*m)*b
        F_mag     = max(F_spring + F_damping, 0)      (hard clamp, no softplus)
        F_normal  = F_mag * n                         (along contact normal)

    Residual:
        F_vec     = F_normal + force_residual         (3-vec correction, unconstrained)
        T_vec     = lever x F_vec
    """

    def __init__(self, input_dim=13, width=256, num_blocks=5,
                 baseline_k=1e3, learn_k=True,
                 baseline_bounciness=0.5, learn_bounciness=True,
                 baseline_mass=1.0, learn_mass=True,
                 head_hidden=64):
        super().__init__()

        # --- backbone ---
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, width),
            nn.LayerNorm(width),
            nn.GELU(),
        )

        backbone_layers = [nn.LayerNorm(width)]
        for _ in range(num_blocks, 1, -1):
            backbone_layers.append(ResBlock(width))
        self.backbone = nn.Sequential(*backbone_layers)

        self.head_trunk = nn.Sequential(
            nn.Linear(width, head_hidden),
            nn.LayerNorm(head_hidden),
            nn.GELU(),
        )
        self.head_out = nn.Linear(head_hidden, HEAD_OUT_DIM)

        # Physics parameters.
        self.k = nn.Parameter(
            torch.tensor(baseline_k, dtype=torch.float32),
            requires_grad=learn_k,
        )
        self.bounciness = nn.Parameter(
            torch.tensor(baseline_bounciness, dtype=torch.float32),
            requires_grad=learn_bounciness,
        )
        self.mass = nn.Parameter(
            torch.tensor(baseline_mass, dtype=torch.float32),
            requires_grad=learn_mass,
        )

    def forward(self, x):
        h = self.input_proj(x)
        h = self.backbone(h)

        raw = self.head_out(self.head_trunk(h))
        collision_logit, depth_raw, force_residual, normal_raw, lever = raw.split(
            [1, 1, 3, 3, 3], dim=-1
        )

        # Match sim: mass clamped to >= 1e-6, k positive.
        k_pos    = F.softplus(self.k)     if self.k.requires_grad else self.k
        mass_pos = F.softplus(self.mass)  if self.mass.requires_grad else self.mass
        mass_pos = torch.clamp(mass_pos, min=1e-6)
        bounciness = torch.sigmoid(self.bounciness)

        # Critical damping
        c = 2.0 * torch.sqrt(k_pos * mass_pos) * bounciness

        # Match sim's `min(penetration, 0.0)`: allow exact zero, clamp positives.
        depth = torch.clamp(depth_raw, max=0.0)

        # Normalize the contact normal (sim does the same defensively).
        contact_normal = F.normalize(normal_raw, dim=-1, eps=1e-8)

        F_spring = -k_pos * depth

        # Damping force along the normal.
        velocity = x[:, VEL_SLICE]
        vel_normal = (velocity * contact_normal).sum(dim=-1, keepdim=True)
        F_damping = -c * vel_normal

        # Sim uses a hard clamp at 0 — never sucks objects into surfaces.
        F_mag = F.relu(F_spring + F_damping)

        # Sim-matching normal force, plus learned residual correction.
        force_normal = F_mag * contact_normal
        force_vec    = force_normal + force_residual
        torque_vec   = torch.cross(lever, force_vec, dim=-1)

        return {
            "collision_logit": collision_logit,
            "force":  force_vec,
            "torque": torque_vec,
            "aux": {
                "contact_normal": contact_normal,
                "lever":          lever,
                "depth":          depth,
                "k":              k_pos.detach(),
                "c":              c.detach(),
                "mass":           mass_pos.detach(),
                "F_mag":          F_mag,
                "F_spring":       F_spring,
                "F_damping":      F_damping,
                "force_normal":   force_normal,
                "force_residual": force_residual,
            },
        }


def make_fast_predictor(input_dim=13, width=256, num_blocks=5, baseline_k=1e3):
    model = WrenchPredictor(input_dim=input_dim,
                            width=width, num_blocks=num_blocks,
                            baseline_k=baseline_k)
    try:
        compiled = torch.compile(model, mode="default")
        print("Model successfully compiled for optimised performance.")
        return compiled
    except Exception as e:
        print(f"torch.compile failed: {e}. Returning standard model.")
        return model

In [50]:
model = make_fast_predictor().to(device)

Model successfully compiled for optimised performance.


Print the model

In [51]:
print(f"Model: {model}")
print(f"Backbone: {model.backbone}")
print(f"Head trunk: {model.head_trunk}")
print(f"Head out: {model.head_out}")

Model: OptimizedModule(
  (_orig_mod): WrenchPredictor(
    (input_proj): Sequential(
      (0): Linear(in_features=13, out_features=256, bias=True)
      (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (2): GELU(approximate='none')
    )
    (backbone): Sequential(
      (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (1): ResBlock(
        (act): ReLU(inplace=True)
        (block): Sequential(
          (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (1): Linear(in_features=256, out_features=1024, bias=True)
          (2): ReLU(inplace=True)
          (3): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (4): Linear(in_features=1024, out_features=256, bias=True)
          (5): ReLU(inplace=True)
        )
      )
      (2): ResBlock(
        (act): ReLU(inplace=True)
        (block): Sequential(
          (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (1): Linear(in_features=256, out_features

Check the model

In [52]:
model.eval()
with torch.no_grad():
    model(torch.zeros([1,13]).to(device))

### Benchmarking

Benchmark the evaluation speed of the model

In [53]:
NUM_BENCHMARKS = 1000

import time
import torch._logging
torch._logging.set_logs(recompiles=True, graph_breaks=True)
x = torch.rand((1, 13), device=device)
results = []

benchmark_model = WrenchPredictor().to(device).eval()
benchmark_model = torch.compile(
    benchmark_model,
    mode="reduce-overhead",
    fullgraph=True,  # fail loudly if the graph has breaks; forces you to fix them
)
benchmark_model.eval()

if torch.cuda.is_available():

    torch.backends.cudnn.benchmark = True
    with torch.inference_mode():

        # warmup
        for _ in tqdm(range(1000), desc="Warmup"):
            benchmark_model(x)
        
        print("=====Warmum completed======")

        for _ in tqdm(range (NUM_BENCHMARKS), desc="Benchmark"):
            torch.cuda.synchronize()
            start = time.perf_counter()
            benchmark_model(x)
            torch.cuda.synchronize()
            end = time.perf_counter()
            results.append((end-start)*1000)
    

    torch.backends.cudnn.benchmark = False

    per_pred_us = sum(results) / NUM_BENCHMARKS
    print(f"Per prediction: {per_pred_us:.2f} ms")

In [54]:
if torch.cuda.is_available():
    print(f"\n=== Benchmark Results ({device}) ===")
    print(f"Max: {max(results)} ms")
    print(f"Min: {min(results)} ms")
    print(f"Avg: {sum(results) / len(results)} ms" )
    print(f"Median: {sorted(results)[len(results) // 2]} ms")
    print("Results:")
    print(results)

## 4. Loss

Huber (smooth-L1) loss replaces L1.  Near zero it's quadratic (smooth gradients,
won't over-punish tiny residuals); far from zero it's linear (robust to the
occasional outlier).  `delta=1.0` is in *normalized* target units so it's
roughly one standard deviation of the target.

**Energy-conservation penalty.**  For each collision sample we integrate one
timestep forward using the *predicted* wrench and check whether the body's
kinetic energy would grow beyond `e^2 * KE_before` (where `e` is the
coefficient of restitution).  Any excess is squared and added to the loss,
so the term is zero for physically admissible predictions and grows smoothly
when the model would inject energy — the exact failure mode you were seeing
at low collision speeds.  You control it with `w_energy`, `dt`, `mass`,
`inertia_diag`, and `restitution` when constructing `WrenchLoss`.


In [55]:
class WrenchLoss(nn.Module):
    """Physics-structured loss for a Hooke (linear elastic) contact model.

    Components:
        - BCE on collision_logit (with pos_weight for class imbalance).
        - Masked Huber on force and torque, operating in PHYSICAL units.
        - Soft penalty on f_n < 0 (Signorini violation).
        - Soft penalty on energy gain during collision (restitution-aware),
          using a BOUNDED log-based term so large violations at init don't
          produce enormous gradients that kill regression learning.

    Energy-conservation term
    ------------------------
    Given a predicted force F and torque tau applied over one timestep dt to a
    body with mass m and (diagonal) inertia I, the post-step velocities are
        v'   = v   + (F   / m) * dt
        w'   = w   + (I^-1 tau) * dt
    and the kinetic energy is  KE = 0.5 m |v|^2 + 0.5 w^T I w.
    For a real collision with coefficient of restitution e in [0, 1] we expect
        KE'  <=  e^2 * KE_before.
    We define the energy ratio
        r = KE' / (e^2 * KE_before)
    and penalise log(max(r, 1))^2. This is zero when r <= 1 (admissible),
    grows like (log r)^2 when r > 1, and its gradient in r is bounded — so a
    random-init network that predicts wildly wrong forces at epoch 0 won't
    produce an exploding energy gradient that pushes the model into the
    degenerate F≈0 basin.

    Warmup: w_energy typically starts at 0 and is ramped up over several
    epochs by the training loop via `set_energy_weight`, so the regression
    heads (force, torque) get to learn first before conservation pressure
    kicks in.

    Because targets are NOT pre-normalised, you may need to set w_force and
    w_torque to bring the two regression terms to comparable magnitude. A good
    heuristic is to set:
        w_force  ~ 1 / (force_rms_in_physical_units)
        w_torque ~ 1 / (torque_rms_in_physical_units)
    so both contribute roughly equally early in training.
    """
    def __init__(self,
                 w_force=1.0, w_torque=1.0, w_collision=1.0,
                 w_energy=0.0,
                 huber_delta=1.0, pos_weight=None,
                 dt=0.24, mass=1.0, inertia_diag=(1.0/6.0, 1.0/6.0, 1.0/6.0),
                 restitution=0.5):
        super().__init__()
        self.w_force     = w_force
        self.w_torque    = w_torque
        self.w_collision = w_collision
        self.w_energy    = w_energy
        self.huber_delta = huber_delta
        self.dt          = float(dt)
        self.mass        = float(mass)
        self.restitution = float(restitution)
        # Inertia tensor (diagonal) for a unit cube by default: I = (1/6) m a^2
        # with m=1, a=1. Override via the constructor to match your simulated body.
        self.register_buffer(
            "inertia_diag",
            torch.tensor(inertia_diag, dtype=torch.float32),
        )
        # pos_weight is a tensor; register as buffer so .to(device) moves it.
        if pos_weight is not None and not torch.is_tensor(pos_weight):
            pos_weight = torch.tensor(float(pos_weight))
        self.register_buffer(
            "pos_weight",
            pos_weight if pos_weight is not None else torch.tensor(1.0),
        )
        self._has_pos_weight = pos_weight is not None

    def set_energy_weight(self, w):
        """Runtime hook for the training loop's warmup schedule."""
        self.w_energy = float(w)

    def _kinetic_energy(self, v, w):
        """KE = 0.5 m |v|^2 + 0.5 w^T I w for diagonal I. Shapes: (B,3)."""
        ke_lin = 0.5 * self.mass * (v * v).sum(dim=-1, keepdim=True)
        ke_rot = 0.5 * (self.inertia_diag * w * w).sum(dim=-1, keepdim=True)
        return ke_lin + ke_rot

    def forward(self, preds, targets):
        mask   = targets["is_collision"]                   # (B, 1)
        n_coll = mask.sum().clamp_min(1.0)

        # --- collision BCE ---
        loss_collision = F.binary_cross_entropy_with_logits(
            preds["collision_logit"], mask.float(),
            pos_weight=self.pos_weight if self._has_pos_weight else None,
            reduction="mean",
        )
        # --- masked Huber on force and torque (physical units) ---
        raw_f = F.huber_loss(preds["force"],  targets["force"],
                             reduction="none", delta=self.huber_delta)  # (B, 3)
        raw_t = F.huber_loss(preds["torque"], targets["torque"],
                             reduction="none", delta=self.huber_delta)  # (B, 3)
        loss_force  = (raw_f.sum(dim=-1, keepdim=True) * mask).sum() / n_coll
        loss_torque = (raw_t.sum(dim=-1, keepdim=True) * mask).sum() / n_coll

        # --- energy-conservation penalty (bounded, log-based) ---
        # Integrate one step using the predicted wrench and compare KE before vs after.
        # Only collision samples contribute (non-contact steps have F=tau=0 anyway).
        v = targets["lin_vel"]                              # (B, 3)
        w = targets["ang_vel"]                              # (B, 3)
        f_pred = preds["force"]                             # (B, 3)
        t_pred = preds["torque"]                            # (B, 3)

        v_next = v + (f_pred / self.mass) * self.dt
        # Diagonal inertia -> element-wise divide
        w_next = w + (t_pred / self.inertia_diag) * self.dt

        ke_before = self._kinetic_energy(v, w)              # (B, 1)
        ke_after  = self._kinetic_energy(v_next, w_next)    # (B, 1)

        # Log-ratio penalty:
        #   r = ke_after / (e^2 * ke_before + eps),  penalty = max(log r, 0)^2
        # Bounded gradient in F: d/dF log(ke_after) scales as 1/ke_after, so at
        # init where ke_after is huge the gradient is SMALL — the opposite of
        # (ke_after - budget)^2, which has gradient proportional to ke_after.
        eps       = 1e-6
        ke_budget = (self.restitution ** 2) * ke_before
        log_ratio = torch.log(ke_after + eps) - torch.log(ke_budget + eps)
        excess_log  = F.relu(log_ratio)                     # zero when admissible
        loss_energy = ((excess_log ** 2) * mask).sum() / n_coll

        total = (self.w_force     * loss_force
               + self.w_torque    * loss_torque
               + self.w_collision * loss_collision
               + self.w_energy    * loss_energy)

        # Diagnostic: fraction of collision samples that currently violate conservation,
        # plus the geometric-mean ratio so you can see HOW MUCH they violate by.
        with torch.no_grad():
            violating = ((excess_log > 0).float() * mask).sum() / n_coll
            # Mean log-ratio over collision samples (in log space so it's well-behaved)
            mean_log_ratio = (log_ratio * mask).sum() / n_coll

        return total, {
            "force":             loss_force.item(),
            "torque":            loss_torque.item(),
            "collision":         loss_collision.item(),
            "energy":            loss_energy.item(),
            "energy_violating":  violating.item(),
            "energy_mean_logr":  mean_log_ratio.item(),
            "w_energy":          self.w_energy,
            "total":             total.item(),
            "active_collisions": n_coll.item(),
            "k":                 preds["aux"]["k"].item(),
        }

## Training

In [56]:
def train_model(model, train_loader, val_loader, train_dataset,
                epochs=200, lr=1e-4, weight_decay=1e-4,
                w_force=1.0, w_torque=1.0, w_collision=0.5,
                # Energy-conservation warmup schedule.
                # Rationale: starting with w_energy>0 produces enormous gradients
                # at init (ke_after is huge for a random-init network) and pushes
                # the model into the degenerate F≈0 basin, where regression loss
                # plateaus at force_rms. Warming up from 0 lets the force/torque
                # heads learn a reasonable solution first; only then do we
                # gently tighten conservation.
                w_energy_max=0.1, energy_warmup_start=20, energy_warmup_epochs=30,
                dt=0.24, mass=1.0, inertia_diag=(1.0/6.0, 1.0/6.0, 1.0/6.0),
                restitution=0.5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Class-imbalance weight for BCE: #no-contact / #contact on train set.
    collisions = train_dataset.collisions.squeeze(-1).bool()
    n_pos = int(collisions.sum().item())
    n_neg = int((~collisions).sum().item())
    pos_weight = (n_neg / max(n_pos, 1)) if n_pos > 0 else 1.0
    print(f"BCE pos_weight = {pos_weight:.3f}  ({n_pos} contacts / {n_neg} non-contacts)")

    criterion = WrenchLoss(
        w_force=w_force, w_torque=w_torque, w_collision=w_collision,
        w_energy=0.0,  # ramped up by the warmup schedule below
        dt=dt, mass=mass, inertia_diag=inertia_diag, restitution=restitution,
        huber_delta=1.0, pos_weight=pos_weight,
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=20
    )

    def energy_weight_at(epoch):
        """Linear ramp from 0 to w_energy_max over [start, start+epochs)."""
        if epoch < energy_warmup_start:
            return 0.0
        if energy_warmup_epochs <= 0:
            return float(w_energy_max)
        frac = (epoch - energy_warmup_start) / float(energy_warmup_epochs)
        return float(w_energy_max) * min(max(frac, 0.0), 1.0)

    best_val_loss = float('inf')
    loss_keys = ['total', 'force', 'torque', 'collision',
                 'energy', 'energy_violating', 'energy_mean_logr']

    for epoch in tqdm(range(epochs)):
        criterion.set_energy_weight(energy_weight_at(epoch))

        # --- Train ---
        model.train()
        train_losses = {k: 0.0 for k in loss_keys}
        for features, targets in train_loader:
            features = features.to(device)
            out = model(features)
            targets  = {k: v.to(device) for k, v in targets.items()}
            optimizer.zero_grad()

            loss, components = criterion(out, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            for k in loss_keys:
                train_losses[k] += components[k]
        for k in loss_keys:
            train_losses[k] /= len(train_loader)

        # --- Validate ---
        model.eval()
        val_losses = {k: 0.0 for k in loss_keys}
        with torch.no_grad():
            for features, targets in val_loader:
                features = features.to(device)
                targets  = {k: v.to(device) for k, v in targets.items()}
                _, components = criterion(model(features), targets)
                for k in loss_keys:
                    val_losses[k] += components[k]
        for k in loss_keys:
            val_losses[k] /= len(val_loader)

        scheduler.step(val_losses['total'])

        if val_losses['total'] < best_val_loss:
            best_val_loss = val_losses['total']
            torch.save({
                'epoch':                epoch,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_loss':        best_val_loss,
            }, 'wrench_model_best.pth')

        if (epoch + 1) % 10 == 0:
            lr_now = optimizer.param_groups[0]['lr']
            k_now  = components.get('k', float('nan'))
            we_now = criterion.w_energy
            print(f"Epoch {epoch+1}/{epochs} | LR: {lr_now:.2e} | k={k_now:.1f} | w_energy={we_now:.4f}")
            print(f"  Train: total={train_losses['total']:.4f} "
                  f"f={train_losses['force']:.4f} t={train_losses['torque']:.4f} "
                  f"c={train_losses['collision']:.4f} "
                  f"e={train_losses['energy']:.4f} vio={train_losses['energy_violating']:.2f} "
                  f"logr={train_losses['energy_mean_logr']:+.3f}")
            print(f"  Val:   total={val_losses['total']:.4f} "
                  f"f={val_losses['force']:.4f} t={val_losses['torque']:.4f} "
                  f"c={val_losses['collision']:.4f} "
                  f"e={val_losses['energy']:.4f} vio={val_losses['energy_violating']:.2f} "
                  f"logr={val_losses['energy_mean_logr']:+.3f}")

    return model

### Execute training

In [ ]:
model = train_model(model, train_loader, val_loader, full_dataset,
                    epochs=200, lr=1e-3,
                    w_force=1.0, w_torque=1.0, w_collision=0.5)


BCE pos_weight = 0.921  (1117730 contacts / 1029467 non-contacts)


  0%|          | 0/800 [00:00<?, ?it/s]V0502 08:26:11.477000 13313 torch/_dynamo/guards.py:4760] [2/1] [__recompiles] Recompiling function forward in /var/folders/fl/x2l8mlls3p5dn1kh36j3rgdm0000gn/T/ipykernel_13313/895762188.py:84
V0502 08:26:11.477000 13313 torch/_dynamo/guards.py:4760] [2/1] [__recompiles]     triggered by the following guard failure(s):
V0502 08:26:11.477000 13313 torch/_dynamo/guards.py:4760] [2/1] [__recompiles]     - 2/0: GLOBAL_STATE changed: grad_mode 
  0%|          | 0/800 [01:39<?, ?it/s]


KeyboardInterrupt: 

## Evaluation

In [ ]:
@torch.no_grad()
def evaluate(model, loader, dataset, device="cuda",
             rel_floor_force=0.05, rel_floor_torque=0.05):
    """Evaluate in physical units (targets were not normalised).

    Args:
        dataset: the *underlying* ContactDataset (not a Subset). Kept for API
                 symmetry; no target stats are needed since targets are physical.
        rel_floor_{force,torque}: targets with magnitude below this (in
                 physical units) are excluded from the relative-error stats
                 to avoid division-by-near-zero blow-up.
    """
    model.eval()
    all_abs_f, all_abs_t = [], []
    all_rel_f, all_rel_t = [], []
    all_coll_correct     = []

    # Physics-diagnostic accumulators (frictionless model: only f_n sign check)
    n_fn_neg = 0
    n_total  = 0

    for features, targets in loader:
        features = features.to(device)
        f_tgt = targets["force"].to(device)
        t_tgt = targets["torque"].to(device)
        c_tgt = targets["is_collision"].to(device).squeeze(-1).bool()

        preds = model(features)
        f_pred = preds["force"]
        t_pred = preds["torque"]
        c_pred = (torch.sigmoid(preds["collision_logit"]).squeeze(-1) > 0.5)

        all_coll_correct.append((c_pred == c_tgt).float().cpu())

        if c_tgt.any():
            f_pred_c = f_pred[c_tgt]
            f_tgt_c  = f_tgt[c_tgt]
            t_pred_c = t_pred[c_tgt]
            t_tgt_c  = t_tgt[c_tgt]

            # Targets are already physical — no de-normalisation needed.
            abs_f = (f_pred_c - f_tgt_c).norm(dim=-1)
            abs_t = (t_pred_c - t_tgt_c).norm(dim=-1)
            all_abs_f.append(abs_f.cpu())
            all_abs_t.append(abs_t.cpu())

            f_norm = f_tgt_c.norm(dim=-1)
            t_norm = t_tgt_c.norm(dim=-1)
            mask_f = f_norm > rel_floor_force
            mask_t = t_norm > rel_floor_torque
            if mask_f.any():
                rel_f = (f_pred_c[mask_f] - f_tgt_c[mask_f]).norm(dim=-1) / f_norm[mask_f]
                all_rel_f.append(rel_f.cpu())
            if mask_t.any():
                rel_t = (t_pred_c[mask_t] - t_tgt_c[mask_t]).norm(dim=-1) / t_norm[mask_t]
                all_rel_t.append(rel_t.cpu())

            # Physics diagnostic: how often the Hooke-plus-residual allows f_n < 0.
            f_n_c = preds["aux"]["force_residual"][c_tgt]
            n_fn_neg += int((f_n_c < 0).sum().item())
            n_total  += int(c_tgt.sum().item())

    abs_f = torch.cat(all_abs_f) if all_abs_f else torch.empty(0)
    abs_t = torch.cat(all_abs_t) if all_abs_t else torch.empty(0)
    rel_f = torch.cat(all_rel_f) if all_rel_f else torch.empty(0)
    rel_t = torch.cat(all_rel_t) if all_rel_t else torch.empty(0)
    coll_acc = torch.cat(all_coll_correct).mean().item()

    def fmt_pct(e):
        if e.numel() == 0:
            return "n/a"
        return (f"p50={e.median():.1%}  p90={e.quantile(0.9):.1%}  "
                f"p99={e.quantile(0.99):.1%}")
    def fmt_abs(e):
        if e.numel() == 0:
            return "n/a"
        return (f"p50={e.median():.4f}  p90={e.quantile(0.9):.4f}  "
                f"p99={e.quantile(0.99):.4f}")

    print("─" * 60)
    print(f"Collision accuracy : {coll_acc:.3%}")
    print(f"Force  abs err     : {fmt_abs(abs_f)}")
    print(f"Torque abs err     : {fmt_abs(abs_t)}")
    print(f"Force  rel err     : {fmt_pct(rel_f)}  "
          f"(on {rel_f.numel()} / {abs_f.numel()} samples above floor)")
    print(f"Torque rel err     : {fmt_pct(rel_t)}  "
          f"(on {rel_t.numel()} / {abs_t.numel()} samples above floor)")
    if n_total > 0:
        print(f"Physics violations : f_n<0 in {n_fn_neg}/{n_total} "
              f"({100*n_fn_neg/n_total:.2f}%)")
    print("─" * 60)

    return {
        "collision_acc":  coll_acc,
        "force_abs_p50":  abs_f.median().item() if abs_f.numel() else float("nan"),
        "force_abs_p99":  abs_f.quantile(0.99).item() if abs_f.numel() else float("nan"),
        "torque_abs_p50": abs_t.median().item() if abs_t.numel() else float("nan"),
        "torque_abs_p99": abs_t.quantile(0.99).item() if abs_t.numel() else float("nan"),
        "fn_neg_rate":    (n_fn_neg / n_total) if n_total > 0 else float("nan"),
    }

## Print 100 data points

In [ ]:
@torch.no_grad()
def print_predictions(model, loader, n=100, device="cuda"):
    """Print n predictions vs ground truth from the loader."""
    model.eval()

    all_f_pred, all_f_tgt = [], []
    all_t_pred, all_t_tgt = [], []
    all_coll_pred, all_coll_tgt = [], []
    all_D, all_power = [], []

    for features, targets in loader:
        features = features.to(device)
        preds = model(features)

        all_f_pred.append(preds["force"].cpu())
        all_t_pred.append(preds["torque"].cpu())
        all_f_tgt.append(targets["force"])
        all_t_tgt.append(targets["torque"])
        all_coll_pred.append(torch.sigmoid(preds["collision_logit"]).cpu())
        all_coll_tgt.append(targets["is_collision"])

        collected = sum(x.shape[0] for x in all_f_pred)
        if collected >= n:
            break

    f_pred = torch.cat(all_f_pred)[:n]
    f_tgt  = torch.cat(all_f_tgt)[:n]
    t_pred = torch.cat(all_t_pred)[:n]
    t_tgt  = torch.cat(all_t_tgt)[:n]
    c_pred = torch.cat(all_coll_pred)[:n].squeeze(-1)
    c_tgt  = torch.cat(all_coll_tgt)[:n].squeeze(-1)

    header = (f"{'#':>4s}  {'coll':>5s} {'pred':>5s}  "
              f"{'force_pred':>30s}  {'force_true':>30s}  "
              f"{'torque_pred':>30s}  {'torque_true':>30s}  ")
    print(header)
    print("─" * len(header))

    for i in range(n):
        cp = f"{c_pred[i]:.2f}"
        ct = f"{int(c_tgt[i].item())}"
        fp = f"[{f_pred[i,0]:8.3f}, {f_pred[i,1]:8.3f}, {f_pred[i,2]:8.3f}]"
        ft = f"[{f_tgt[i,0]:8.3f}, {f_tgt[i,1]:8.3f}, {f_tgt[i,2]:8.3f}]"
        tp = f"[{t_pred[i,0]:8.4f}, {t_pred[i,1]:8.4f}, {t_pred[i,2]:8.4f}]"
        tt = f"[{t_tgt[i,0]:8.4f}, {t_tgt[i,1]:8.4f}, {t_tgt[i,2]:8.4f}]"
        print(f"{i:4d}  {ct:>5s} {cp:>5s}  {fp:>30s}  {ft:>30s}  {tp:>30s}  {tt:>30s}")

print_predictions(model, val_loader, n=100)


AssertionError: Torch not compiled with CUDA enabled

Download checkpoint (Colab)

In [ ]:
if is_colab():
    from google.colab import files
    files.download("wrench_model_best.pth")